# GPU 버전

In [1]:
# !pip install natsort

In [2]:
import torch

torch.cuda.is_available()

True

In [3]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(torch.cuda.device_count())

NVIDIA GeForce RTX 5090
1


# 데이터셋 선택 및 하이퍼파라미터 설정
▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼

In [4]:
import time
import natsort
import os

#folder_list = os.listdir("./data/")
folder_list = ['eraser']
item_list = natsort.natsorted(folder_list)

print("다음 데이터셋들이 학습됩니다 : ", item_list)

다음 데이터셋들이 학습됩니다 :  ['eraser']


In [5]:
#최소10, 200~400 추천, 10단위로 pth가 저장됨
epochs = 200
batch_size = 16

#3090 24GB에서 64까지 사용 가능했음
learning_rate = 0.005
image_size = 256

▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲

# Main.py

In [6]:
import torch
from dataset import get_data_transforms
from torchvision.datasets import ImageFolder
import numpy as np
import random
import os
from torch.utils.data import DataLoader
from resnet import resnet18, resnet34, resnet50, wide_resnet50_2
from de_resnet import de_resnet18, de_resnet34, de_wide_resnet50_2, de_resnet50
from dataset import RD_Dataset
import torch.backends.cudnn as cudnn
import argparse
from torch.nn import functional as F

In [7]:
def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [8]:
def loss_fucntion(a, b):
    #mse_loss = torch.nn.MSELoss()
    cos_loss = torch.nn.CosineSimilarity()
    loss = 0
    for item in range(len(a)):
        #print(a[item].shape)
        #print(b[item].shape)
        #loss += 0.1*mse_loss(a[item], b[item])
        loss += torch.mean(1-cos_loss(a[item].view(a[item].shape[0],-1),
                                      b[item].view(b[item].shape[0],-1)))
    return loss

In [9]:
def loss_concat(a, b):
    mse_loss = torch.nn.MSELoss()
    cos_loss = torch.nn.CosineSimilarity()
    loss = 0
    a_map = []
    b_map = []
    size = a[0].shape[-1]
    for item in range(len(a)):
        #loss += mse_loss(a[item], b[item])
        a_map.append(F.interpolate(a[item], size=size, mode='bilinear', align_corners=True))
        b_map.append(F.interpolate(b[item], size=size, mode='bilinear', align_corners=True))
    a_map = torch.cat(a_map,1)
    b_map = torch.cat(b_map,1)
    loss += torch.mean(1-cos_loss(a_map,b_map))
    return loss

In [10]:
def train(_class_):
    print(_class_)
        
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(device)

    data_transform = get_data_transforms(image_size, image_size)
    
    train_path = './data/' + _class_ + '/train'
    ckp_path = './checkpoints/' + 'wres50_'+_class_+'.pth'
    os.makedirs('./checkpoints', exist_ok=True)
    
    train_data = ImageFolder(root=train_path, transform=data_transform)
    train_dataloader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)

    encoder, bn = wide_resnet50_2(pretrained=True)
    encoder = encoder.to(device)
    bn = bn.to(device)
    encoder.eval()
    decoder = de_wide_resnet50_2(pretrained=False)
    decoder = decoder.to(device)

    optimizer = torch.optim.Adam(list(decoder.parameters())+list(bn.parameters()), lr=learning_rate, betas=(0.5,0.999))


    for epoch in range(epochs):
        start = time.time() 
        
        bn.train()
        decoder.train()
        loss_list = []
        for img, label in train_dataloader:
            img = img.to(device)
            inputs = encoder(img)
            outputs = decoder(bn(inputs))#bn(inputs))
            loss = loss_fucntion(inputs, outputs)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_list.append(loss.item())
        print('epoch [{}/{}], loss:{:.4f}'.format(epoch + 1, epochs, np.mean(loss_list)))
        print("time :",time.time() - start)  # 현재시각 - 시작시간 = 실행 시간
        
        if (epoch + 1) % 10 == 0:
            torch.save({'bn': bn.state_dict(),'decoder': decoder.state_dict()}, ckp_path)
            
    return loss

# 학습 시작

In [11]:
setup_seed(111)

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [12]:
#학습
for i in item_list:
    start_class = time.time()  # 시작 시간 저장

    train(i)
    print(i, "time :",time.time() - start_class)  # 현재시각 - 시작시간 = 실행 시간

eraser
cuda
epoch [1/200], loss:1.7606
time : 0.59794020652771
epoch [2/200], loss:1.4157
time : 0.2236483097076416
epoch [3/200], loss:1.0781
time : 0.18974947929382324
epoch [4/200], loss:0.9045
time : 0.20094704627990723
epoch [5/200], loss:0.7812
time : 0.20290899276733398
epoch [6/200], loss:0.7149
time : 0.192702054977417
epoch [7/200], loss:0.6733
time : 0.19390225410461426
epoch [8/200], loss:0.6419
time : 0.20798110961914062
epoch [9/200], loss:0.6038
time : 0.19348669052124023
epoch [10/200], loss:0.5805
time : 0.18588876724243164
epoch [11/200], loss:0.5687
time : 0.1953723430633545
epoch [12/200], loss:0.5355
time : 0.19605159759521484
epoch [13/200], loss:0.5117
time : 0.20301318168640137
epoch [14/200], loss:0.5023
time : 0.19236016273498535
epoch [15/200], loss:0.4863
time : 0.19312000274658203
epoch [16/200], loss:0.4889
time : 0.18593668937683105
epoch [17/200], loss:0.4715
time : 0.1979050636291504
epoch [18/200], loss:0.4363
time : 0.19499635696411133
epoch [19/200],

KeyboardInterrupt: 